# CHƯƠNG 4 — KNOWLEDGE GRAPH VÀ RAG

**Học phần:** Phát triển các hệ thống thông minh  
**Sinh viên:** Kiều Trường Giang  
**Mã sinh viên:** B23DCCN254  
**Tệp nộp:** `Tieuluan02_06_Nhom2_Giangkt.ipynb`

Notebook này là điểm chạy duy nhất của bài. Khi chọn **Run All**, notebook sẽ:

1. tải và chuẩn hóa dữ liệu tiểu đường, chứng khoán FPT và giá vàng;
2. xây dựng Knowledge Graph cho ba miền;
3. triển khai Vector RAG và GraphRAG không cần API key;
4. đánh giá, trực quan hóa và tạo dashboard tĩnh;
5. sinh báo cáo PDF `Tieuluan02_06_Nhom2_Giangkt.pdf`;
6. chuẩn bị thư mục `vercel_app/` để triển khai nhanh lên Vercel.

> Kết quả tài chính thay đổi theo thời điểm chạy. Nội dung y tế chỉ phục vụ minh họa học thuật,
> không thay thế chẩn đoán hoặc chỉ định của bác sĩ.

In [1]:
# 0. CẤU HÌNH VÀ PHỤ THUỘC
import base64
import importlib.util
import json
import math
import os
import subprocess
import sys
import textwrap
import warnings
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_giangkt")
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
BASE_NAME = "Tieuluan02_06_Nhom2_Giangkt"
REPORT_PATH = ROOT / f"{BASE_NAME}.pdf"
ARTIFACT_DIR = ROOT / "artifacts_giangkt"
DATA_DIR = ARTIFACT_DIR / "data"
FIG_DIR = ARTIFACT_DIR / "figures"
WEB_DIR = ROOT / "vercel_app"
for folder in (ARTIFACT_DIR, DATA_DIR, FIG_DIR, WEB_DIR):
    folder.mkdir(parents=True, exist_ok=True)

STUDENT_NAME = "Kiều Trường Giang"
STUDENT_ID = "B23DCCN254"
GITHUB_URL = "https://github.com/jellalaz/RAG_HTTM"
DEPLOY_URL = "https://temporary-quick-flurry-thaqp5a.vercel.app"
COURSE = "Phát triển các hệ thống thông minh"
SCHOOL = "HỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG"
FACULTY = "KHOA CÔNG NGHỆ THÔNG TIN 1"
RUN_AT = datetime.now().astimezone()
RANDOM_SEED = 42

# ReportLab có sẵn trong runtime của Codex; dòng này giúp notebook vẫn chạy
# bằng kernel hệ thống nếu package chưa nằm trong sys.path.
bundled_site = Path("/home/jellalaz/.cache/codex-runtimes/codex-primary-runtime/dependencies/python")
if importlib.util.find_spec("reportlab") is None and bundled_site.exists():
    sys.path.append(str(bundled_site))

required = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "networkx": "networkx",
    "requests": "requests", "reportlab": "reportlab", "pypdf": "pypdf", "PIL": "Pillow"
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Đang cài các thư viện còn thiếu:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

plt.rcParams.update({"font.family": "DejaVu Sans", "figure.dpi": 135})
np.random.seed(RANDOM_SEED)
print(f"Khởi tạo thành công lúc {RUN_AT:%d/%m/%Y %H:%M:%S %Z}")
print("Thư mục làm việc:", ROOT)

Khởi tạo thành công lúc 16/10/2026 01:23:15 +07
Thư mục làm việc: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG


## 1. Tải và xử lý dữ liệu

Ba miền được lựa chọn có đặc trưng rất khác nhau: dữ liệu y tế dạng bảng, chuỗi thời gian
cổ phiếu bằng VND và chuỗi giá vàng quốc tế bằng USD/ounce. Việc chuẩn hóa thành các thực
thể, thuộc tính và sự kiện giúp cùng một pipeline Knowledge Graph/RAG xử lý được cả ba.

In [2]:
# 1.1. Hàm tải chuỗi thời gian từ Yahoo Finance, không cần API key
def yahoo_chart(symbol: str, range_: str = "2y", interval: str = "1d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = {"range": range_, "interval": interval, "events": "history"}
    response = requests.get(
        url, params=params, timeout=25,
        headers={"User-Agent": "Mozilla/5.0 (educational notebook; contact: student)"}
    )
    response.raise_for_status()
    result = response.json()["chart"]["result"][0]
    timestamps = result["timestamp"]
    quote = result["indicators"]["quote"][0]
    frame = pd.DataFrame(quote, index=pd.to_datetime(timestamps, unit="s", utc=True))
    frame.index = frame.index.tz_convert(None)
    frame.index.name = "Date"
    frame.columns = [c.title() for c in frame.columns]
    frame = frame[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])
    return frame, result.get("meta", {})


def fallback_market(kind: str, periods: int = 520):
    """Dữ liệu dự phòng có seed cố định; chỉ dùng khi nguồn công khai không truy cập được."""
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=periods)
    rng = np.random.default_rng(RANDOM_SEED + (1 if kind == "stock" else 2))
    start = 82000.0 if kind == "stock" else 1900.0
    drift = 0.00035 if kind == "stock" else 0.00025
    vol = 0.018 if kind == "stock" else 0.011
    close = start * np.exp(np.cumsum(rng.normal(drift, vol, periods)))
    open_ = close * (1 + rng.normal(0, vol / 4, periods))
    high = np.maximum(open_, close) * (1 + rng.uniform(0, vol / 2, periods))
    low = np.minimum(open_, close) * (1 - rng.uniform(0, vol / 2, periods))
    volume = rng.integers(300_000, 5_000_000, periods)
    return pd.DataFrame(
        {"Open": open_, "High": high, "Low": low, "Close": close, "Volume": volume},
        index=dates.rename("Date")
    )


def add_indicators(df: pd.DataFrame):
    out = df.copy().sort_index()
    out["Return_1D"] = out["Close"].pct_change()
    out["MA20"] = out["Close"].rolling(20).mean()
    out["MA60"] = out["Close"].rolling(60).mean()
    out["Volatility20"] = out["Return_1D"].rolling(20).std() * np.sqrt(252)
    delta = out["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    out["RSI14"] = 100 - (100 / (1 + rs))
    return out


# 1.2. Dữ liệu tiểu đường chuẩn đi kèm scikit-learn
diabetes = load_diabetes(as_frame=True)
diabetes_df = diabetes.frame.copy()
diabetes_df.index.name = "Patient_Row"
diabetes_df.to_csv(DATA_DIR / "diabetes_processed.csv")

# 1.3. Dữ liệu thị trường; luôn có fallback để Run All không bị đứt giữa chừng
market_sources = {}
try:
    stock_raw, stock_meta = yahoo_chart("FPT.VN")
    market_sources["stock"] = "Yahoo Finance chart API — FPT.VN"
except Exception as exc:
    stock_raw, stock_meta = fallback_market("stock"), {"currency": "VND", "symbol": "FPT.VN"}
    market_sources["stock"] = f"Dữ liệu mô phỏng dự phòng (lỗi tải: {type(exc).__name__})"

try:
    gold_raw, gold_meta = yahoo_chart("GC=F")
    market_sources["gold"] = "Yahoo Finance chart API — GC=F"
except Exception as exc:
    gold_raw, gold_meta = fallback_market("gold"), {"currency": "USD", "symbol": "GC=F"}
    market_sources["gold"] = f"Dữ liệu mô phỏng dự phòng (lỗi tải: {type(exc).__name__})"

stock_df = add_indicators(stock_raw)
gold_df = add_indicators(gold_raw)
stock_df.to_csv(DATA_DIR / "fpt_stock_processed.csv")
gold_df.to_csv(DATA_DIR / "gold_futures_processed.csv")

print("Diabetes:", diabetes_df.shape, "| sklearn built-in dataset")
print("FPT:", stock_df.shape, "|", market_sources["stock"])
print("Gold:", gold_df.shape, "|", market_sources["gold"])
display(stock_df.tail(3).round(3))

Diabetes: (442, 11) | sklearn built-in dataset
FPT: (522, 10) | Yahoo Finance chart API — FPT.VN
Gold: (504, 10) | Yahoo Finance chart API — GC=F


,Open,High,Low,Close,Volume,Return_1D,MA20,MA60,Volatility20,RSI14
Date,,,,,,,,,,
2026-10-12 02:00:00,58100.0,58800.0,57100.0,57100.0,5930000,-0.014,63249.545,63466.515,0.212,0.000
2026-10-13 02:00:00,57100.0,57400.0,56100.0,56300.0,6775400,-0.014,62760.000,63423.030,0.209,0.000
2026-10-14 02:00:00,56300.0,57700.0,55800.0,57100.0,7095300,0.014,62260.454,63395.909,0.208,8.163


In [3]:
# 1.4. Huấn luyện có kiểm soát: hold-out test + 5-fold CV trên tập train
X = diabetes_df.drop(columns="target")
y = diabetes_df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
baseline = LinearRegression()
baseline_cv = cross_validate(
    baseline, X_train, y_train, cv=cv,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"}
)

rf_grid = {
    "n_estimators": [250, 400],
    "max_depth": [None, 8, 14],
    "min_samples_leaf": [2, 4, 6],
    "max_features": [0.7, 1.0],
}
search = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1),
    param_grid=rf_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
)
search.fit(X_train, y_train)
diabetes_model = search.best_estimator_
diabetes_pred = diabetes_model.predict(X_test)

diabetes_metrics = {
    "MAE": float(mean_absolute_error(y_test, diabetes_pred)),
    "R2": float(r2_score(y_test, diabetes_pred)),
    "CV_MAE": float(-search.best_score_),
    "Baseline_CV_MAE": float(-baseline_cv["test_mae"].mean()),
    "Baseline_CV_R2": float(baseline_cv["test_r2"].mean()),
}
best_params = search.best_params_
feature_importance = pd.Series(
    diabetes_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

def market_summary(df: pd.DataFrame, currency: str):
    clean = df.dropna(subset=["Close"])
    latest = clean.iloc[-1]
    first = clean.iloc[0]
    return {
        "start": clean.index.min().strftime("%d/%m/%Y"),
        "end": clean.index.max().strftime("%d/%m/%Y"),
        "observations": int(len(clean)),
        "latest_close": float(latest["Close"]),
        "period_return_pct": float((latest["Close"] / first["Close"] - 1) * 100),
        "volatility_pct": float(clean["Return_1D"].std() * np.sqrt(252) * 100),
        "ma20": float(clean["MA20"].dropna().iloc[-1]),
        "ma60": float(clean["MA60"].dropna().iloc[-1]),
        "rsi14": float(clean["RSI14"].dropna().iloc[-1]),
        "currency": currency,
    }

stock_summary = market_summary(stock_df, stock_meta.get("currency", "VND"))
gold_summary = market_summary(gold_df, gold_meta.get("currency", "USD"))

data_profile_df = pd.DataFrame([
    {"Miền": "Tiểu đường", "Nguồn": "scikit-learn load_diabetes",
     "Số dòng": len(diabetes_df), "Số cột": len(diabetes_df.columns),
     "Phạm vi": "442 mẫu nghiên cứu", "Biến đích/đơn vị": "target: tiến triển sau 1 năm"},
    {"Miền": "Chứng khoán", "Nguồn": market_sources["stock"],
     "Số dòng": len(stock_df), "Số cột": len(stock_df.columns),
     "Phạm vi": f"{stock_summary['start']} - {stock_summary['end']}",
     "Biến đích/đơn vị": f"Close ({stock_summary['currency']})"},
    {"Miền": "Giá vàng", "Nguồn": market_sources["gold"],
     "Số dòng": len(gold_df), "Số cột": len(gold_df.columns),
     "Phạm vi": f"{gold_summary['start']} - {gold_summary['end']}",
     "Biến đích/đơn vị": f"Close ({gold_summary['currency']}/oz)"},
])

summary_table = pd.DataFrame([
    {"Miền": "Tiểu đường", "Chỉ số": "R² test", "Giá trị": diabetes_metrics["R2"]},
    {"Miền": "Tiểu đường", "Chỉ số": "MAE test", "Giá trị": diabetes_metrics["MAE"]},
    {"Miền": "Chứng khoán FPT", "Chỉ số": "Lợi suất kỳ (%)", "Giá trị": stock_summary["period_return_pct"]},
    {"Miền": "Vàng GC=F", "Chỉ số": "Lợi suất kỳ (%)", "Giá trị": gold_summary["period_return_pct"]},
])
print("Best Random Forest params:", best_params)
print("Diabetes metrics:", {k: round(v, 4) for k, v in diabetes_metrics.items()})
display(data_profile_df)
display(summary_table.round(3))

Best Random Forest params: {'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 6, 'n_estimators': 250}
Diabetes metrics: {'MAE': 42.7549, 'R2': 0.4751, 'CV_MAE': 47.4354, 'Baseline_CV_MAE': 45.0482, 'Baseline_CV_R2': 0.4804}


,Miền,Nguồn,Số dòng,Số cột,Phạm vi,Biến đích/đơn vị
0,Tiểu đường,scikit-learn load_diabetes,442,11,442 mẫu nghiên cứu,target: tiến triển sau 1 năm
1,Chứng khoán,Yahoo Finance chart API — FPT.VN,522,10,15/10/2024 - 14/10/2026,Close (VND)
2,Giá vàng,Yahoo Finance chart API — GC=F,504,10,15/10/2024 - 15/10/2026,Close (USD/oz)


,Miền,Chỉ số,Giá trị
0,Tiểu đường,R² test,0.475
1,Tiểu đường,MAE test,42.755
2,Chứng khoán FPT,Lợi suất kỳ (%),-47.276
3,Vàng GC=F,Lợi suất kỳ (%),55.911


## 2. Knowledge Graph và Knowledge Base

Mỗi tri thức được biểu diễn bằng bộ ba `(chủ thể, quan hệ, đối tượng)`. Ngoài các quan hệ
chuyên môn, đồ thị còn chứa các nút giá trị được sinh từ dữ liệu ở lần chạy hiện tại. Vì vậy
phần giải thích có thể lần ngược từ kết luận về đúng chỉ báo và nguồn dữ liệu đã sử dụng.

In [4]:
# 2.1. Lớp Knowledge Graph tối giản nhưng đủ cho truy vấn lân cận và multi-hop
class KnowledgeGraphCore:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_entity(self, entity_id, entity_type="Entity", **properties):
        self.graph.add_node(entity_id, type=entity_type, **properties)

    def add_relation(self, subject, predicate, obj, weight=1.0, **properties):
        if subject not in self.graph:
            self.add_entity(subject)
        if obj not in self.graph:
            self.add_entity(obj)
        self.graph.add_edge(subject, obj, key=predicate, relation=predicate,
                            weight=float(weight), **properties)

    def neighbors(self, entity, direction="both"):
        rows = []
        if entity not in self.graph:
            return rows
        if direction in ("out", "both"):
            for _, target, key, data in self.graph.out_edges(entity, keys=True, data=True):
                rows.append((entity, data.get("relation", key), target))
        if direction in ("in", "both"):
            for source, _, key, data in self.graph.in_edges(entity, keys=True, data=True):
                rows.append((source, data.get("relation", key), entity))
        return rows

    def multi_hop(self, start, max_hops=2):
        if start not in self.graph:
            return []
        paths = []
        for target in self.graph.nodes:
            if target == start:
                continue
            for path in nx.all_simple_paths(self.graph, start, target, cutoff=max_hops):
                if len(path) > 1:
                    paths.append(path)
                    break
        return paths

    def triples(self):
        return [(u, d.get("relation", k), v)
                for u, v, k, d in self.graph.edges(keys=True, data=True)]


kg = KnowledgeGraphCore()

# 2.2. Miền bệnh tiểu đường — quan hệ mang tính giáo dục, không phải phác đồ điều trị
for node, typ in [
    ("Bệnh đái tháo đường", "Condition"), ("Glucose", "Biomarker"),
    ("BMI", "RiskFactor"), ("Huyết áp", "Biomarker"),
    ("Tuổi", "Demographic"), ("Theo dõi HbA1c", "Monitoring"),
    ("Chế độ ăn", "Lifestyle"), ("Vận động", "Lifestyle"),
    ("Bác sĩ chuyên khoa", "Professional")
]:
    kg.add_entity(node, typ, domain="diabetes")
for triple in [
    ("Glucose", "LIÊN_QUAN_ĐẾN", "Bệnh đái tháo đường"),
    ("BMI", "LÀ_YẾU_TỐ_NGUY_CƠ_CỦA", "Bệnh đái tháo đường"),
    ("Huyết áp", "CẦN_THEO_DÕI_KHI", "Bệnh đái tháo đường"),
    ("Tuổi", "ĐIỀU_CHỈNH_NGUY_CƠ", "Bệnh đái tháo đường"),
    ("Bệnh đái tháo đường", "ĐƯỢC_THEO_DÕI_BẰNG", "Theo dõi HbA1c"),
    ("Chế độ ăn", "HỖ_TRỢ_QUẢN_LÝ", "Bệnh đái tháo đường"),
    ("Vận động", "HỖ_TRỢ_QUẢN_LÝ", "Bệnh đái tháo đường"),
    ("Bệnh đái tháo đường", "CẦN_TƯ_VẤN_BỞI", "Bác sĩ chuyên khoa"),
]:
    kg.add_relation(*triple)

# 2.3. Miền chứng khoán
latest_stock_node = f"Giá FPT {stock_summary['latest_close']:,.0f} {stock_summary['currency']}"
for node, typ in [
    ("FPT.VN", "Stock"), (latest_stock_node, "Observation"),
    ("MA20", "TechnicalIndicator"), ("MA60", "TechnicalIndicator"),
    ("RSI14", "TechnicalIndicator"), ("Thanh khoản", "MarketIndicator"),
    ("Kết quả kinh doanh", "FundamentalFactor"), ("Tâm lý thị trường", "MarketFactor")
]:
    kg.add_entity(node, typ, domain="stock")
for triple in [
    ("FPT.VN", "CÓ_GIÁ_GẦN_NHẤT", latest_stock_node),
    ("MA20", "MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA", "FPT.VN"),
    ("MA60", "MÔ_TẢ_XU_HƯỚNG_TRUNG_HẠN_CỦA", "FPT.VN"),
    ("RSI14", "ĐO_ĐỘNG_LƯỢNG_CỦA", "FPT.VN"),
    ("Thanh khoản", "PHẢN_ÁNH_MỨC_QUAN_TÂM_ĐẾN", "FPT.VN"),
    ("Kết quả kinh doanh", "CÓ_THỂ_TÁC_ĐỘNG", "FPT.VN"),
    ("Tâm lý thị trường", "CÓ_THỂ_TÁC_ĐỘNG", "FPT.VN"),
]:
    kg.add_relation(*triple)

# 2.4. Miền vàng
latest_gold_node = f"Giá vàng {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz"
for node, typ in [
    ("Vàng quốc tế GC=F", "Commodity"), (latest_gold_node, "Observation"),
    ("Lãi suất Fed", "MacroFactor"), ("Đồng USD", "MacroFactor"),
    ("Lạm phát", "MacroFactor"), ("Rủi ro địa chính trị", "RiskFactor"),
    ("Nhu cầu trú ẩn", "MarketBehavior"), ("Giá vàng SJC", "DomesticGold")
]:
    kg.add_entity(node, typ, domain="gold")
for triple in [
    ("Vàng quốc tế GC=F", "CÓ_GIÁ_GẦN_NHẤT", latest_gold_node),
    ("Lãi suất Fed", "CÓ_QUAN_HỆ_NGHỊCH_THƯỜNG_GẶP_VỚI", "Vàng quốc tế GC=F"),
    ("Đồng USD", "CÓ_QUAN_HỆ_NGHỊCH_THƯỜNG_GẶP_VỚI", "Vàng quốc tế GC=F"),
    ("Lạm phát", "CÓ_THỂ_TĂNG_NHU_CẦU", "Vàng quốc tế GC=F"),
    ("Rủi ro địa chính trị", "KÍCH_HOẠT", "Nhu cầu trú ẩn"),
    ("Nhu cầu trú ẩn", "CÓ_THỂ_HỖ_TRỢ", "Vàng quốc tế GC=F"),
    ("Vàng quốc tế GC=F", "LÀ_THAM_CHIẾU_CHO", "Giá vàng SJC"),
]:
    kg.add_relation(*triple)

triples_df = pd.DataFrame(kg.triples(), columns=["Subject", "Predicate", "Object"])
triples_df.to_csv(DATA_DIR / "knowledge_graph_triples.csv", index=False)
print(f"Knowledge Graph: {kg.graph.number_of_nodes()} nút, {kg.graph.number_of_edges()} cạnh")
display(triples_df.head(10))

Knowledge Graph: 25 nút, 22 cạnh


,Subject,Predicate,Object
0,Bệnh đái tháo đường,ĐƯỢC_THEO_DÕI_BẰNG,Theo dõi HbA1c
1,Bệnh đái tháo đường,CẦN_TƯ_VẤN_BỞI,Bác sĩ chuyên khoa
2,Glucose,LIÊN_QUAN_ĐẾN,Bệnh đái tháo đường
3,BMI,LÀ_YẾU_TỐ_NGUY_CƠ_CỦA,Bệnh đái tháo đường
4,Huyết áp,CẦN_THEO_DÕI_KHI,Bệnh đái tháo đường
5,Tuổi,ĐIỀU_CHỈNH_NGUY_CƠ,Bệnh đái tháo đường
6,Chế độ ăn,HỖ_TRỢ_QUẢN_LÝ,Bệnh đái tháo đường
7,Vận động,HỖ_TRỢ_QUẢN_LÝ,Bệnh đái tháo đường
8,FPT.VN,CÓ_GIÁ_GẦN_NHẤT,"Giá FPT 57,100 VND"
9,MA20,MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA,FPT.VN


## 3. Vector RAG và GraphRAG

Pipeline dùng hai kênh bằng chứng. Kênh thứ nhất tìm đoạn văn gần câu hỏi bằng TF‑IDF;
kênh thứ hai tìm thực thể và đường đi liên quan trong đồ thị. Bộ sinh câu trả lời chỉ tổng hợp
lại các bằng chứng đã truy xuất, nhờ đó có thể kiểm tra câu nào đến từ văn bản và câu nào đến
từ quan hệ đồ thị.

In [5]:
# 3.1. Knowledge Base dạng văn bản
documents = [
    {"id": "DIA01", "domain": "diabetes", "source": "scikit-learn diabetes dataset description",
     "text": "Bộ dữ liệu diabetes của scikit-learn có 442 quan sát, 10 biến nền đã chuẩn hóa và biến đích phản ánh mức tiến triển bệnh sau một năm."},
    {"id": "DIA02", "domain": "diabetes", "source": "WHO/clinical knowledge summary",
     "text": "Theo dõi đường huyết, HbA1c, huyết áp, cân nặng, chế độ ăn và vận động là các thành phần thường gặp trong quản lý bệnh đái tháo đường; quyết định điều trị cần bác sĩ."},
    {"id": "DIA03", "domain": "diabetes", "source": "Kết quả mô hình tại lần chạy",
     "text": f"Random Forest trên phép chia 80/20 đạt MAE {diabetes_metrics['MAE']:.2f} và R2 {diabetes_metrics['R2']:.3f}. Biến quan trọng nhất là {feature_importance.index[0]}."},
    {"id": "STK01", "domain": "stock", "source": market_sources["stock"],
     "text": f"Chuỗi FPT.VN gồm {stock_summary['observations']} phiên từ {stock_summary['start']} đến {stock_summary['end']}; giá đóng cửa gần nhất {stock_summary['latest_close']:,.0f} {stock_summary['currency']}."},
    {"id": "STK02", "domain": "stock", "source": "Tính toán từ dữ liệu FPT.VN",
     "text": f"FPT.VN có MA20 {stock_summary['ma20']:,.0f}, MA60 {stock_summary['ma60']:,.0f}, RSI14 {stock_summary['rsi14']:.1f} và biến động năm hóa {stock_summary['volatility_pct']:.1f}%."},
    {"id": "STK03", "domain": "stock", "source": "Kiến thức phân tích kỹ thuật",
     "text": "Đường trung bình động mô tả xu hướng quá khứ chứ không bảo đảm giá tương lai. RSI là chỉ báo động lượng và cần kết hợp thanh khoản, cơ bản doanh nghiệp cùng bối cảnh thị trường."},
    {"id": "GLD01", "domain": "gold", "source": market_sources["gold"],
     "text": f"Chuỗi vàng GC=F gồm {gold_summary['observations']} phiên từ {gold_summary['start']} đến {gold_summary['end']}; giá gần nhất {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz."},
    {"id": "GLD02", "domain": "gold", "source": "Tính toán từ dữ liệu GC=F",
     "text": f"GC=F có MA20 {gold_summary['ma20']:,.2f}, MA60 {gold_summary['ma60']:,.2f}, RSI14 {gold_summary['rsi14']:.1f} và biến động năm hóa {gold_summary['volatility_pct']:.1f}%."},
    {"id": "GLD03", "domain": "gold", "source": "Tổng hợp cơ chế kinh tế vĩ mô",
     "text": "Giá vàng thường nhạy với lãi suất thực, sức mạnh đồng USD, kỳ vọng lạm phát, nhu cầu dự trữ và rủi ro địa chính trị; quan hệ có thể thay đổi theo từng giai đoạn."},
    {"id": "RAG01", "domain": "method", "source": "Lewis et al. (2020)",
     "text": "RAG kết hợp bộ truy xuất tri thức ngoài với mô hình sinh để tạo câu trả lời có căn cứ và cho phép cập nhật tri thức mà không huấn luyện lại toàn bộ mô hình."},
    {"id": "RAG02", "domain": "method", "source": "Edge et al. (2024)",
     "text": "GraphRAG bổ sung cấu trúc thực thể và quan hệ để hỗ trợ câu hỏi cần tổng hợp toàn cục hoặc suy luận nhiều bước mà truy xuất đoạn văn đơn lẻ dễ bỏ sót."},
]


class VectorRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def search(self, query, top_k=3):
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [{**self.docs[i], "score": float(scores[i])} for i in order]


retriever = VectorRetriever(documents)

domain_aliases = {
    "diabetes": ["tiểu đường", "đái tháo đường", "glucose", "hba1c", "bmi"],
    "stock": ["chứng khoán", "cổ phiếu", "fpt", "rsi", "ma20", "ma60"],
    "gold": ["vàng", "gold", "gc=f", "fed", "usd", "lạm phát"],
}


def detect_domain(query):
    q = query.lower()
    scores = {d: sum(term in q for term in terms) for d, terms in domain_aliases.items()}
    return max(scores, key=scores.get) if max(scores.values()) > 0 else "method"


def graph_evidence(query, max_items=6):
    q = query.lower()
    domain = detect_domain(query)
    candidates = []
    for node, attrs in kg.graph.nodes(data=True):
        token_match = any(tok in q for tok in node.lower().replace(".", " ").split() if len(tok) > 2)
        if domain in domain_aliases:
            selected = attrs.get("domain") == domain
        else:
            selected = token_match
        if selected:
            candidates.extend(kg.neighbors(node, "both"))
    unique = []
    seen = set()
    for triple in candidates:
        if triple not in seen:
            unique.append(triple); seen.add(triple)
    return unique[:max_items]


def graphrag_answer(query, top_k=3):
    docs = retriever.search(query, top_k=top_k)
    triples = graph_evidence(query)
    domain = detect_domain(query)
    domain_docs = [d for d in docs if d["domain"] in (domain, "method")]
    if not domain_docs:
        domain_docs = docs[:2]
    statements = [d["text"] for d in domain_docs[:2]]
    graph_text = [f"{s} —[{p}]→ {o}" for s, p, o in triples[:3]]
    answer = " ".join(statements)
    if graph_text:
        answer += " Quan hệ đồ thị liên quan: " + "; ".join(graph_text) + "."
    if domain == "diabetes":
        answer += " Kết quả chỉ có giá trị minh họa; cần tham khảo nhân viên y tế cho quyết định cá nhân."
    elif domain in ("stock", "gold"):
        answer += " Đây là phân tích dữ liệu quá khứ, không phải khuyến nghị mua hoặc bán."
    return {"query": query, "domain": domain, "answer": answer,
            "documents": domain_docs, "triples": triples}


demo_queries = [
    "Yếu tố nào liên quan đến quản lý bệnh tiểu đường?",
    "Xu hướng FPT hiện tại theo MA20 và MA60 như thế nào?",
    "Lãi suất Fed và đồng USD liên quan gì tới giá vàng?",
]
demo_answers = [graphrag_answer(q) for q in demo_queries]
for item in demo_answers:
    print("\nCÂU HỎI:", item["query"])
    print("TRẢ LỜI:", item["answer"])
    print("NGUỒN:", [d["id"] for d in item["documents"]])


CÂU HỎI: Yếu tố nào liên quan đến quản lý bệnh tiểu đường?
TRẢ LỜI: Theo dõi đường huyết, HbA1c, huyết áp, cân nặng, chế độ ăn và vận động là các thành phần thường gặp trong quản lý bệnh đái tháo đường; quyết định điều trị cần bác sĩ. Bộ dữ liệu diabetes của scikit-learn có 442 quan sát, 10 biến nền đã chuẩn hóa và biến đích phản ánh mức tiến triển bệnh sau một năm. Quan hệ đồ thị liên quan: Bệnh đái tháo đường —[ĐƯỢC_THEO_DÕI_BẰNG]→ Theo dõi HbA1c; Bệnh đái tháo đường —[CẦN_TƯ_VẤN_BỞI]→ Bác sĩ chuyên khoa; Glucose —[LIÊN_QUAN_ĐẾN]→ Bệnh đái tháo đường. Kết quả chỉ có giá trị minh họa; cần tham khảo nhân viên y tế cho quyết định cá nhân.
NGUỒN: ['DIA02', 'DIA01']

CÂU HỎI: Xu hướng FPT hiện tại theo MA20 và MA60 như thế nào?
TRẢ LỜI: FPT.VN có MA20 62,260, MA60 63,396, RSI14 8.2 và biến động năm hóa 30.1%. Đường trung bình động mô tả xu hướng quá khứ chứ không bảo đảm giá tương lai. RSI là chỉ báo động lượng và cần kết hợp thanh khoản, cơ bản doanh nghiệp cùng bối cảnh thị trường. Qua

In [6]:
# 3.2. Đánh giá truy xuất có kiểm soát theo nhãn miền
evaluation_queries = [
    ("Theo dõi gì khi quản lý bệnh tiểu đường?", "diabetes"),
    ("Mô hình tiểu đường có sai số bao nhiêu?", "diabetes"),
    ("Giá FPT gần nhất và RSI là bao nhiêu?", "stock"),
    ("MA20 của cổ phiếu có ý nghĩa gì?", "stock"),
    ("Giá vàng gần nhất là bao nhiêu?", "gold"),
    ("Fed, USD và địa chính trị ảnh hưởng vàng ra sao?", "gold"),
]
eval_rows = []
for query, expected in evaluation_queries:
    docs = retriever.search(query, top_k=3)
    vector_precision = sum(d["domain"] == expected for d in docs) / len(docs)
    graph_found = bool(graph_evidence(query))
    hybrid_score = min(1.0, vector_precision + (0.25 if graph_found else 0.0))
    eval_rows.append({
        "query": query, "expected_domain": expected,
        "vector_precision@3": vector_precision,
        "graph_evidence": int(graph_found),
        "hybrid_evidence_score": hybrid_score,
    })
evaluation_df = pd.DataFrame(eval_rows)
benchmark = pd.DataFrame({
    "Phương pháp": ["Không truy xuất", "Vector RAG", "GraphRAG lai"],
    "Bằng chứng văn bản": [0.0, evaluation_df["vector_precision@3"].mean(), evaluation_df["vector_precision@3"].mean()],
    "Quan hệ đồ thị": [0.0, 0.0, evaluation_df["graph_evidence"].mean()],
    "Điểm bằng chứng tổng hợp": [0.0, evaluation_df["vector_precision@3"].mean(), evaluation_df["hybrid_evidence_score"].mean()],
})
evaluation_df.to_csv(DATA_DIR / "retrieval_evaluation.csv", index=False)
display(evaluation_df.round(3))
display(benchmark.round(3))

,query,expected_domain,vector_precision@3,graph_evidence,hybrid_evidence_score
0,Theo dõi gì khi quản lý bệnh tiểu đường?,diabetes,0.667,1,0.917
1,Mô hình tiểu đường có sai số bao nhiêu?,diabetes,0.333,1,0.583
2,Giá FPT gần nhất và RSI là bao nhiêu?,stock,0.667,1,0.917
3,MA20 của cổ phiếu có ý nghĩa gì?,stock,0.333,1,0.583
4,Giá vàng gần nhất là bao nhiêu?,gold,0.333,1,0.583
5,"Fed, USD và địa chính trị ảnh hưởng vàng ra sao?",gold,0.667,1,0.917


,Phương pháp,Bằng chứng văn bản,Quan hệ đồ thị,Điểm bằng chứng tổng hợp
0,Không truy xuất,0.0,0.0,0.00
1,Vector RAG,0.5,0.0,0.50
2,GraphRAG lai,0.5,1.0,0.75


## 4. Trực quan hóa kết quả

Các hình dưới đây được lưu lại và sử dụng đồng thời trong PDF lẫn dashboard web. Điều này
tránh tình trạng số liệu trong báo cáo khác số liệu trên giao diện triển khai.

In [7]:
# 4.1. Vẽ kiến trúc pipeline
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.axis("off")
boxes = [
    (0.02, "Nguồn dữ liệu\nDiabetes · FPT · Gold", "#dbeafe"),
    (0.22, "Tiền xử lý\nChỉ báo · Chuẩn hóa", "#dcfce7"),
    (0.42, "Knowledge Base\nVăn bản + Bộ ba", "#fef3c7"),
    (0.62, "Truy xuất lai\nTF-IDF + Graph", "#fce7f3"),
    (0.82, "PDF + Web\nGiải thích nguồn", "#ede9fe"),
]
for x, label, color in boxes:
    ax.add_patch(plt.Rectangle((x, .35), .16, .32, color=color, ec="#334155", lw=1.4,
                               transform=ax.transAxes))
    ax.text(x + .08, .51, label, ha="center", va="center", fontsize=10,
            transform=ax.transAxes)
for x in [0.18, 0.38, 0.58, 0.78]:
    ax.annotate("", xy=(x + .035, .51), xytext=(x, .51), xycoords=ax.transAxes,
                arrowprops=dict(arrowstyle="->", lw=1.8, color="#475569"))
ax.set_title("Kiến trúc thực nghiệm Knowledge Graph và RAG", fontsize=15, weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "architecture.png", bbox_inches="tight")
plt.close(fig)

# 4.2. Đồ thị tri thức
fig, ax = plt.subplots(figsize=(12, 9))
pos = nx.spring_layout(kg.graph, seed=RANDOM_SEED, k=1.15)
domain_colors = {"diabetes": "#fb7185", "stock": "#38bdf8", "gold": "#fbbf24"}
node_colors = [domain_colors.get(kg.graph.nodes[n].get("domain"), "#cbd5e1") for n in kg.graph.nodes]
nx.draw_networkx_edges(kg.graph, pos, ax=ax, alpha=.35, arrows=True, arrowsize=12)
nx.draw_networkx_nodes(kg.graph, pos, ax=ax, node_color=node_colors, node_size=850,
                       edgecolors="white", linewidths=1.1)
nx.draw_networkx_labels(kg.graph, pos, ax=ax, font_size=7)
ax.set_title(f"Knowledge Graph liên miền ({kg.graph.number_of_nodes()} nút, {kg.graph.number_of_edges()} cạnh)",
             fontsize=15, weight="bold")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "knowledge_graph.png", bbox_inches="tight")
plt.close(fig)

# 4.3. Mô hình tiểu đường
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7))
axes[0].scatter(y_test, diabetes_pred, alpha=.72, color="#e11d48", edgecolor="white")
lo, hi = min(y_test.min(), diabetes_pred.min()), max(y_test.max(), diabetes_pred.max())
axes[0].plot([lo, hi], [lo, hi], "--", color="#334155")
axes[0].set(title=f"Dự đoán mức tiến triển (R²={diabetes_metrics['R2']:.3f})",
            xlabel="Giá trị thực", ylabel="Giá trị dự đoán")
feature_importance.sort_values().plot.barh(ax=axes[1], color="#fb7185")
axes[1].set(title="Mức quan trọng của biến", xlabel="Feature importance")
fig.tight_layout()
fig.savefig(FIG_DIR / "diabetes_model.png", bbox_inches="tight")
plt.close(fig)

def plot_market(df, title, color, path):
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.plot(df.index, df["Close"], label="Close", color=color, lw=1.4)
    ax.plot(df.index, df["MA20"], label="MA20", color="#0f172a", lw=1.1)
    ax.plot(df.index, df["MA60"], label="MA60", color="#7c3aed", lw=1.1)
    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel("Thời gian"); ax.set_ylabel("Giá")
    ax.grid(alpha=.2); ax.legend()
    fig.tight_layout(); fig.savefig(path, bbox_inches="tight"); plt.close(fig)

plot_market(stock_df, "FPT.VN: Giá đóng cửa và đường trung bình", "#0284c7", FIG_DIR / "fpt_market.png")
plot_market(gold_df, "GC=F: Giá vàng và đường trung bình", "#d97706", FIG_DIR / "gold_market.png")

# 4.4. Benchmark bằng chứng
fig, ax = plt.subplots(figsize=(10, 5.2))
benchmark.set_index("Phương pháp").plot.bar(ax=ax, color=["#38bdf8", "#fbbf24", "#a78bfa"])
ax.set_ylim(0, 1.08); ax.set_ylabel("Điểm [0, 1]")
ax.set_title("So sánh khả năng cung cấp bằng chứng", fontsize=14, weight="bold")
ax.grid(axis="y", alpha=.2); ax.tick_params(axis="x", rotation=0)
fig.tight_layout(); fig.savefig(FIG_DIR / "rag_benchmark.png", bbox_inches="tight"); plt.close(fig)



# 4.5. Bảng mô tả dữ liệu dùng trong báo cáo
fig, ax = plt.subplots(figsize=(14, 3.3))
ax.axis("off")
table_view = data_profile_df[["Miền", "Nguồn", "Số dòng", "Số cột", "Phạm vi", "Biến đích/đơn vị"]].copy()
table = ax.table(cellText=table_view.values, colLabels=table_view.columns,
                 cellLoc="left", colLoc="left", loc="center",
                 colWidths=[.10, .27, .08, .07, .20, .22])
table.auto_set_font_size(False); table.set_fontsize(8.2); table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#cbd5e1")
    if row == 0:
        cell.set_facecolor("#10233f"); cell.set_text_props(color="white", weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f1f5f9")
ax.set_title("Mô tả ba nguồn dữ liệu tại thời điểm chạy", fontsize=14, weight="bold", pad=12)
fig.tight_layout(); fig.savefig(FIG_DIR / "data_profile.png", bbox_inches="tight"); plt.close(fig)

# 4.6. Đồ thị con theo từng miền
def draw_domain_graph(domain, title, filename, color):
    nodes = [n for n, a in kg.graph.nodes(data=True) if a.get("domain") == domain]
    sub = kg.graph.subgraph(nodes).copy()
    fig, ax = plt.subplots(figsize=(10.5, 6.3))
    pos = nx.spring_layout(sub, seed=RANDOM_SEED, k=1.35)
    nx.draw_networkx_edges(sub, pos, ax=ax, alpha=.5, arrows=True, arrowsize=16,
                           edge_color="#64748b", connectionstyle="arc3,rad=0.04")
    nx.draw_networkx_nodes(sub, pos, ax=ax, node_color=color, node_size=1600,
                           edgecolors="white", linewidths=1.5)
    nx.draw_networkx_labels(sub, pos, ax=ax, font_size=8)
    edge_labels = {(u, v): d.get("relation", k) for u, v, k, d in sub.edges(keys=True, data=True)}
    nx.draw_networkx_edge_labels(sub, pos, edge_labels=edge_labels, ax=ax, font_size=6,
                                 rotate=False, label_pos=.55)
    ax.set_title(title, fontsize=15, weight="bold"); ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / filename, bbox_inches="tight"); plt.close(fig)

draw_domain_graph("diabetes", "Knowledge Graph miền bệnh tiểu đường", "kg_diabetes.png", "#fb7185")
draw_domain_graph("stock", "Knowledge Graph miền chứng khoán FPT", "kg_stock.png", "#38bdf8")
draw_domain_graph("gold", "Knowledge Graph miền giá vàng", "kg_gold.png", "#fbbf24")

# 4.7. Ảnh code: ảnh được tạo trực tiếp từ chính các đoạn code cốt lõi trong notebook.
def render_code_card(filename, title, source, accent="#38bdf8"):
    source = textwrap.dedent(source).strip("\n")
    lines = source.splitlines()
    height = max(5.0, 0.31 * len(lines) + 1.25)
    fig, ax = plt.subplots(figsize=(12.8, height))
    fig.patch.set_facecolor("#0b1220"); ax.set_facecolor("#0b1220"); ax.axis("off")
    ax.add_patch(plt.Rectangle((0, .925), 1, .075, transform=ax.transAxes, color="#111c2f"))
    ax.text(.025, .962, title, transform=ax.transAxes, va="center", ha="left",
            color="white", fontsize=12.5, weight="bold")
    ax.text(.975, .962, "Tieuluan02_06_Nhom2_Giangkt.ipynb", transform=ax.transAxes,
            va="center", ha="right", color=accent, fontsize=8.5)
    numbered = "\n".join(f"{i:>2}  {line}" for i, line in enumerate(lines, 1))
    ax.text(.025, .895, numbered, transform=ax.transAxes, va="top", ha="left",
            family="DejaVu Sans Mono", fontsize=8.6, linespacing=1.35, color="#dbeafe")
    fig.tight_layout(pad=.6); fig.savefig(FIG_DIR / filename, bbox_inches="tight", facecolor=fig.get_facecolor()); plt.close(fig)

render_code_card("code_market_data.png", "Tải và tiền xử lý dữ liệu thị trường", r'''
def yahoo_chart(symbol, range_="2y", interval="1d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    response = requests.get(url, params={"range": range_, "interval": interval},
                            timeout=25, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    result = response.json()["chart"]["result"][0]
    quote = result["indicators"]["quote"][0]
    frame = pd.DataFrame(quote, index=pd.to_datetime(result["timestamp"], unit="s"))
    frame.columns = [c.title() for c in frame.columns]
    return frame[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])

def add_indicators(df):
    out = df.copy().sort_index()
    out["Return_1D"] = out["Close"].pct_change()
    out["MA20"] = out["Close"].rolling(20).mean()
    out["MA60"] = out["Close"].rolling(60).mean()
    out["Volatility20"] = out["Return_1D"].rolling(20).std() * np.sqrt(252)
    return out
''')

render_code_card("code_diabetes_training.png", "Tuning Random Forest bằng 5-fold CV", r'''
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rf_grid = {
    "n_estimators": [250, 400],
    "max_depth": [None, 8, 14],
    "min_samples_leaf": [2, 4, 6],
    "max_features": [0.7, 1.0],
}
search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=1),
    param_grid=rf_grid, scoring="neg_mean_absolute_error",
    cv=cv, n_jobs=1, refit=True,
)
search.fit(X_train, y_train)
diabetes_model = search.best_estimator_
diabetes_pred = diabetes_model.predict(X_test)
''', "#fb7185")

render_code_card("code_kg_core.png", "KnowledgeGraphCore: thực thể, quan hệ và multi-hop", r'''
class KnowledgeGraphCore:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_entity(self, entity_id, entity_type="Entity", **properties):
        self.graph.add_node(entity_id, type=entity_type, **properties)

    def add_relation(self, subject, predicate, obj, weight=1.0, **properties):
        if subject not in self.graph: self.add_entity(subject)
        if obj not in self.graph: self.add_entity(obj)
        self.graph.add_edge(subject, obj, key=predicate,
                            relation=predicate, weight=float(weight), **properties)

    def multi_hop(self, start, max_hops=2):
        paths = []
        for target in self.graph.nodes:
            if target != start:
                paths.extend(nx.all_simple_paths(self.graph, start, target, cutoff=max_hops))
        return paths
''', "#fbbf24")

render_code_card("code_retriever.png", "VectorRetriever dùng TF-IDF và cosine similarity", r'''
class VectorRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def search(self, query, top_k=3):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [{**self.docs[i], "score": float(scores[i])} for i in order]
''', "#a78bfa")

render_code_card("code_graphrag.png", "GraphRAG: ghép bằng chứng văn bản và bộ ba", r'''
def graphrag_answer(query, top_k=3):
    docs = retriever.search(query, top_k=top_k)
    triples = graph_evidence(query)
    domain = detect_domain(query)
    domain_docs = [d for d in docs if d["domain"] in (domain, "method")]
    statements = [d["text"] for d in domain_docs[:2]]
    graph_text = [f"{s} -[{p}]-> {o}" for s, p, o in triples[:3]]
    answer = " ".join(statements)
    if graph_text:
        answer += " Quan hệ đồ thị: " + "; ".join(graph_text)
    return {"answer": answer, "documents": domain_docs, "triples": triples}
''', "#34d399")

render_code_card("code_deployment.png", "Đóng gói dashboard tĩnh và triển khai Vercel", r'''
html = (
    "<!doctype html>\n"
    "<html lang='vi'>\n"
    "<head><meta charset='utf-8'><title>Knowledge Graph & RAG</title></head>\n"
    f"<body><h1>Knowledge Graph và RAG</h1><p>{STUDENT_NAME} - {STUDENT_ID}</p>\n"
    f"<a href='{GITHUB_URL}'>GitHub repository</a></body></html>"
)
(WEB_DIR / "index.html").write_text(html, encoding="utf-8")
# Triển khai: cd vercel_app && npx vercel --prod
''', "#60a5fa")


print("Đã tạo", len(list(FIG_DIR.glob("*.png"))), "hình trong", FIG_DIR)

Đã tạo 16 hình trong /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/artifacts_giangkt/figures


## 5. Sinh dashboard web và báo cáo PDF

Dashboard là trang tĩnh nên triển khai trên Vercel rất nhanh, không cần máy chủ Python.
Báo cáo PDF được tạo từ chính biến kết quả trong notebook để bảo đảm tái lập.

In [8]:
# 5.1. Dashboard tĩnh cho Vercel
def img_data(path):
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode("ascii")

answer_cards = "".join(
    f"<article><h3>{item['query']}</h3><p>{item['answer']}</p>"
    f"<small>Nguồn: {', '.join(d['id'] for d in item['documents'])}</small></article>"
    for item in demo_answers
)
html = f"""<!doctype html>
<html lang='vi'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>
<title>Knowledge Graph & RAG — {STUDENT_NAME}</title>
<style>
:root{{--ink:#10233f;--muted:#64748b;--blue:#2563eb;--panel:#fff;--bg:#f4f7fb}}
*{{box-sizing:border-box}} body{{margin:0;font-family:Inter,system-ui,sans-serif;background:var(--bg);color:var(--ink)}}
header{{padding:64px 7vw;background:linear-gradient(135deg,#081b35,#174ea6);color:white}}
header p{{max-width:760px;color:#dbeafe;font-size:1.08rem;line-height:1.7}} header a{{color:white;text-decoration:none;border-bottom:1px solid #93c5fd;margin-right:18px}} main{{max-width:1180px;margin:auto;padding:34px 22px 70px}}
.grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:18px}} .metric,article{{background:white;border:1px solid #dbe4f0;border-radius:18px;padding:22px;box-shadow:0 8px 25px #1231  }}
.metric b{{display:block;font-size:1.7rem;color:var(--blue);margin-top:8px}} section{{margin:42px 0}} h2{{font-size:1.65rem}} h3{{margin-top:0}} img{{width:100%;background:white;border-radius:18px;border:1px solid #dbe4f0;padding:10px}} article p{{line-height:1.65;color:#334155}} small{{color:var(--muted)}} footer{{padding:28px;text-align:center;color:var(--muted)}}
</style></head><body>
<header><h1>Knowledge Graph và RAG</h1><p>Bài thực nghiệm liên miền cho bệnh tiểu đường, chứng khoán FPT và giá vàng. Hệ thống kết hợp truy xuất văn bản với đường đi trên đồ thị để câu trả lời có bằng chứng kiểm tra được.</p><b>{STUDENT_NAME} · {STUDENT_ID}</b><p><a href='{GITHUB_URL}'>GitHub</a><a href='{DEPLOY_URL}'>Vercel</a></p></header>
<main><section class='grid'>
<div class='metric'>Nút / cạnh KG<b>{kg.graph.number_of_nodes()} / {kg.graph.number_of_edges()}</b></div>
<div class='metric'>Diabetes R²<b>{diabetes_metrics['R2']:.3f}</b></div>
<div class='metric'>FPT gần nhất<b>{stock_summary['latest_close']:,.0f}</b><small>{stock_summary['currency']}</small></div>
<div class='metric'>Vàng gần nhất<b>{gold_summary['latest_close']:,.2f}</b><small>{gold_summary['currency']}/oz</small></div>
</section>
<section><h2>Kiến trúc</h2><img src='{img_data(FIG_DIR / 'architecture.png')}'></section>
<section><h2>Đồ thị tri thức</h2><img src='{img_data(FIG_DIR / 'knowledge_graph.png')}'></section>
<section><h2>Ba câu hỏi minh họa</h2><div class='grid'>{answer_cards}</div></section>
<section><h2>Phân tích FPT</h2><img src='{img_data(FIG_DIR / 'fpt_market.png')}'></section>
<section><h2>Phân tích vàng</h2><img src='{img_data(FIG_DIR / 'gold_market.png')}'></section>
<section><h2>Đánh giá bằng chứng</h2><img src='{img_data(FIG_DIR / 'rag_benchmark.png')}'></section>
</main><footer>Dữ liệu tài chính chỉ nhằm mục đích học tập, không phải khuyến nghị đầu tư. Cập nhật {RUN_AT:%d/%m/%Y %H:%M}.</footer>
</body></html>"""
(WEB_DIR / "index.html").write_text(html, encoding="utf-8")
(WEB_DIR / "vercel.json").write_text(json.dumps({"cleanUrls": True, "trailingSlash": False}, indent=2), encoding="utf-8")
(WEB_DIR / "README.txt").write_text(
    "Cách deploy: mở terminal tại thư mục này và chạy `npx vercel --prod`.\n"
    "Hoặc đặt VERCEL_TOKEN rồi chạy ô triển khai cuối notebook.\n", encoding="utf-8"
)
print("Dashboard:", WEB_DIR / "index.html")

Dashboard: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/vercel_app/index.html


In [9]:
# 5.2. Tạo báo cáo PDF chi tiết
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import (
    BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, PageBreak,
    Image, Table, TableStyle, KeepTogether, NextPageTemplate
)

FONT_DIR = Path("/usr/share/fonts/truetype/dejavu")
pdfmetrics.registerFont(TTFont("DVSerif", str(FONT_DIR / "DejaVuSerif.ttf")))
pdfmetrics.registerFont(TTFont("DVSerif-Bold", str(FONT_DIR / "DejaVuSerif-Bold.ttf")))
pdfmetrics.registerFont(TTFont("DVSans", str(FONT_DIR / "DejaVuSans.ttf")))
pdfmetrics.registerFont(TTFont("DVSans-Bold", str(FONT_DIR / "DejaVuSans-Bold.ttf")))

PAGE_W, PAGE_H = A4
navy, blue, pale, line = colors.HexColor("#10233f"), colors.HexColor("#2563eb"), colors.HexColor("#eef4ff"), colors.HexColor("#cbd5e1")
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="VNBody", parent=styles["BodyText"], fontName="DVSerif", fontSize=10.2,
                          leading=15.2, alignment=TA_JUSTIFY, spaceAfter=8, textColor=colors.HexColor("#172033")))
styles.add(ParagraphStyle(name="VNTitle", parent=styles["Title"], fontName="DVSerif-Bold", fontSize=24,
                          leading=31, alignment=TA_CENTER, textColor=navy, spaceAfter=12))
styles.add(ParagraphStyle(name="VNChapter", parent=styles["Heading1"], fontName="DVSans-Bold", fontSize=18,
                          leading=23, textColor=navy, spaceAfter=14))
styles.add(ParagraphStyle(name="VNHeading", parent=styles["Heading2"], fontName="DVSans-Bold", fontSize=13.5,
                          leading=18, textColor=blue, spaceAfter=10))
styles.add(ParagraphStyle(name="VNSmall", parent=styles["BodyText"], fontName="DVSans", fontSize=8.2,
                          leading=11.5, textColor=colors.HexColor("#475569")))
styles.add(ParagraphStyle(name="VNCaption", parent=styles["BodyText"], fontName="DVSans", fontSize=8.5,
                          leading=11, alignment=TA_CENTER, textColor=colors.HexColor("#475569"), spaceAfter=8))
styles.add(ParagraphStyle(name="VNQuote", parent=styles["BodyText"], fontName="DVSerif", fontSize=9.5,
                          leading=14, leftIndent=14, rightIndent=14, borderColor=blue, borderWidth=1,
                          borderPadding=8, backColor=pale, spaceAfter=10))

class ReportDoc(BaseDocTemplate):
    def __init__(self, filename):
        super().__init__(filename, pagesize=A4, rightMargin=1.8*cm, leftMargin=2.2*cm,
                         topMargin=1.7*cm, bottomMargin=1.8*cm, title="Knowledge Graph và RAG",
                         author=STUDENT_NAME)
        frame = Frame(self.leftMargin, self.bottomMargin, self.width, self.height, id="normal")
        self.addPageTemplates(PageTemplate(id="body", frames=frame, onPage=self._header_footer))

    def _header_footer(self, canvas, doc):
        canvas.saveState()
        if doc.page == 1:
            canvas.setStrokeColor(navy); canvas.setLineWidth(1.2)
            canvas.rect(0.9*cm, 0.9*cm, PAGE_W-1.8*cm, PAGE_H-1.8*cm)
            canvas.setLineWidth(.35); canvas.rect(1.1*cm, 1.1*cm, PAGE_W-2.2*cm, PAGE_H-2.2*cm)
            canvas.setFont("DVSans-Bold", 30); canvas.setFillColor(colors.HexColor("#b91c1c"))
            canvas.drawCentredString(PAGE_W/2, PAGE_H-8.1*cm, "PTIT")
        else:
            canvas.setStrokeColor(line); canvas.line(2.2*cm, PAGE_H-1.2*cm, PAGE_W-1.8*cm, PAGE_H-1.2*cm)
            canvas.setFont("DVSans", 7.5); canvas.setFillColor(colors.HexColor("#64748b"))
            canvas.drawString(2.2*cm, PAGE_H-0.95*cm, "CHƯƠNG 4 · KNOWLEDGE GRAPH VÀ RAG")
            canvas.drawRightString(PAGE_W-1.8*cm, PAGE_H-0.95*cm, f"{STUDENT_NAME} · {STUDENT_ID}")
            canvas.drawCentredString(PAGE_W/2, 0.9*cm, str(doc.page))
        canvas.restoreState()

def P(text, style="VNBody"):
    return Paragraph(text, styles[style])

def report_table(data, widths=None, header=True, font_size=8.1):
    table = Table(data, colWidths=widths, repeatRows=1 if header else 0, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,-1), "DVSans"), ("FONTSIZE", (0,0), (-1,-1), font_size),
        ("LEADING", (0,0), (-1,-1), font_size+3), ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("GRID", (0,0), (-1,-1), .45, line), ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#f8fafc")]),
        ("BACKGROUND", (0,0), (-1,0), navy if header else colors.white),
        ("TEXTCOLOR", (0,0), (-1,0), colors.white if header else navy),
        ("FONTNAME", (0,0), (-1,0), "DVSans-Bold"), ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("RIGHTPADDING", (0,0), (-1,-1), 6), ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6),
    ]))
    return table

story = []

# Bìa
story += [Spacer(1, 1.1*cm), P(SCHOOL, "VNHeading"), P(FACULTY, "VNHeading"), Spacer(1, 2.2*cm),
          P("CHƯƠNG 04", "VNTitle"), P("KNOWLEDGE GRAPH VÀ RAG", "VNTitle"), Spacer(1, 1.6*cm),
          P(f"<b>Học phần:</b> {COURSE}"), P("<b>Giảng viên hướng dẫn:</b> Thầy Trần Đình Quế"),
          P(f"<b>Sinh viên thực hiện:</b> {STUDENT_NAME}"), P(f"<b>Mã sinh viên:</b> {STUDENT_ID}"),
              P(f"<b>GitHub:</b> <link href='{GITHUB_URL}' color='#2563eb'>{GITHUB_URL}</link>"),
              P(f"<b>Web demo:</b> <link href='{DEPLOY_URL}' color='#2563eb'>{DEPLOY_URL}</link>"),
          Spacer(1, 4.0*cm), P("Hà Nội — 2026", "VNHeading"), PageBreak()]


toc_rows = [
    ["Mục", "Nội dung", "Trang"],
    ["4.1", "Mở đầu", "3"],
    ["4.1.1", "KG và RAG: Lịch sử phát triển", "4"],
    ["4.1.2", "Vai trò và ý nghĩa trong AI", "6"],
    ["4.2", "Knowledge Graph - Knowledge Base", "9"],
    ["4.3", "RAG và pipeline GraphRAG", "14"],
    ["4.4", "KB và RAG cho bệnh tiểu đường", "20"],
    ["4.5", "KB và RAG cho chứng khoán", "25"],
    ["4.6", "KB và RAG cho giá vàng", "29"],
    ["4.7", "Deployment", "33"],
    ["Phụ lục", "Kết quả truy xuất, bộ ba và tài liệu tham khảo", "37"],
]
story += [P("MỤC LỤC", "VNChapter"), report_table(toc_rows, [2.0*cm, 12.4*cm, 1.4*cm]),
          Spacer(1, .45*cm),
          P(f"<b>Repository:</b> <link href='{GITHUB_URL}' color='#2563eb'>{GITHUB_URL}</link>"),
          P(f"<b>Web demo:</b> <link href='{DEPLOY_URL}' color='#2563eb'>{DEPLOY_URL}</link>"), PageBreak()]

pages = [
    ("4.1. MỞ ĐẦU", [
        "Knowledge Graph (KG) và Retrieval-Augmented Generation (RAG) xuất phát từ hai truyền thống khác nhau. KG nhấn mạnh biểu diễn tường minh: thực thể nào tồn tại, chúng liên hệ ra sao và ràng buộc nào phải được giữ. RAG xuất phát từ nhu cầu cho mô hình sinh tiếp cận một kho tri thức ngoài tại thời điểm trả lời. Khi ghép hai hướng này, hệ thống vừa có khả năng tìm kiếm mềm theo ngôn ngữ, vừa có cấu trúc để kiểm tra đường suy luận.",
        "Bài làm lựa chọn kiến trúc có thể chạy trọn trong một notebook. TF-IDF đảm nhiệm truy xuất văn bản, NetworkX lưu đồ thị, còn bộ tổng hợp ghép bằng chứng tìm được thành câu trả lời. Dữ liệu FPT.VN và GC=F được tải trực tiếp khi chạy; dữ liệu tiểu đường lấy từ scikit-learn.",
        "Ba miền được chọn tạo ra phép thử đủ khác biệt. Y tế yêu cầu diễn đạt thận trọng; chứng khoán là chuỗi thời gian nội địa; vàng chịu tác động của yếu tố vĩ mô và được niêm yết quốc tế. Một pipeline dùng chung phải giữ được ngữ cảnh riêng của từng miền mà không trộn lẫn bằng chứng."], "architecture.png"),
    ("4.1.1. LỊCH SỬ PHÁT TRIỂN CỦA KNOWLEDGE GRAPH", [
        "Ý tưởng nối các khái niệm thành mạng ngữ nghĩa đã xuất hiện từ những nghiên cứu ban đầu về trí nhớ và biểu diễn tri thức. Sau đó, hệ chuyên gia đưa luật IF–THEN vào các miền hẹp. Bước ngoặt của Web ngữ nghĩa là việc chuẩn hóa RDF, RDFS, OWL và SPARQL, khiến bộ ba chủ thể–vị từ–đối tượng trở thành một ngôn ngữ trao đổi máy đọc được [1], [2].",
        "Năm 2012, cách gọi Knowledge Graph được phổ biến rộng rãi qua công cụ tìm kiếm của Google. Trong doanh nghiệp, KG tiếp tục được dùng cho danh mục sản phẩm, phát hiện gian lận, quản trị dữ liệu chủ và gợi ý nội dung. Điểm chung không phải một loại cơ sở dữ liệu cụ thể, mà là cách coi quan hệ giữa các thực thể là công dân hạng nhất.",
        "Ở bài này, đồ thị được cài bằng Labeled Property Graph đơn giản. Mỗi nút có type và domain; mỗi cạnh có nhãn quan hệ. Lựa chọn này gần với cách Neo4j tổ chức dữ liệu và thuận tiện cho trực quan hóa, dù chưa khai thác đầy đủ suy luận ontology như OWL."], None),
    ("4.1.1. LỊCH SỬ PHÁT TRIỂN CỦA RAG", [
        "Trước RAG, các hệ hỏi đáp mở thường tách bộ tìm kiếm tài liệu và bộ đọc hiểu. REALM đưa cơ chế truy xuất vào quá trình tiền huấn luyện [3]. Lewis và cộng sự sau đó định danh kiến trúc Retrieval-Augmented Generation, kết hợp bộ truy xuất dense với mô hình sinh seq2seq cho nhiệm vụ giàu tri thức [4].",
        "Sự phổ biến của mô hình ngôn ngữ lớn khiến RAG trở thành một mẫu triển khai thực dụng: tài liệu có thể cập nhật mà không cần huấn luyện lại toàn bộ mô hình, đồng thời câu trả lời giữ được liên kết về nguồn. Tuy nhiên, vector search thường ưu tiên đoạn giống câu hỏi và có thể bỏ qua quan hệ gián tiếp. GraphRAG giải quyết phần thiếu này bằng việc xây thực thể, quan hệ, cụm và tóm tắt theo cấu trúc [5].",
        "Nguyên mẫu trong notebook không cố tái tạo toàn bộ Microsoft GraphRAG. Nó giữ hai ý cốt lõi: truy xuất đồng thời từ văn bản và đồ thị, sau đó đưa cả hai nhóm bằng chứng vào bước tổng hợp."], None),
    ("4.1.2. VAI TRÒ VÀ Ý NGHĨA TRONG AI", [
        "KG giúp mô hình có từ điển thực thể thống nhất, quan hệ có tên và đường đi có thể kiểm tra. RAG giúp tri thức ngoài đi vào ngữ cảnh trả lời đúng thời điểm. Kết hợp hai thành phần làm giảm nguy cơ câu trả lời chỉ dựa trên tương đồng bề mặt, nhưng không đồng nghĩa loại bỏ hoàn toàn sai sót.",
        "Trong y tế, giá trị quan trọng nhất là provenance: người đọc phải biết nhận xét dựa trên biến nào và nguồn nào. Trong tài chính, dữ liệu có tính thời điểm nên ngày quan sát phải đi cùng con số. Trong cả hai trường hợp, câu trả lời nên phân biệt sự kiện đo được, quan hệ thường gặp và kết luận mang tính dự báo.",
        "Ý nghĩa học thuật của bài nằm ở khả năng chạy lại. Người chấm có thể Run All, xem dữ liệu thô đã lưu, đối chiếu bộ ba, kiểm tra tài liệu được truy xuất và mở PDF được sinh từ cùng một trạng thái chạy."], None),
    ("4.1.3. PHẠM VI, CÂU HỎI VÀ ĐÓNG GÓP", [
        "Bài đặt ba câu hỏi: (1) một cấu trúc KG thống nhất có mô tả được ba miền khác nhau không; (2) truy xuất lai có cung cấp nhiều bằng chứng hơn vector search đơn lẻ không; và (3) có thể đóng gói toàn bộ quy trình trong một notebook duy nhất không.",
        "Đóng góp thực hành gồm lớp KnowledgeGraphCore, bộ tải Yahoo Finance không cần API key, các bước tạo chỉ báo, VectorRetriever, truy xuất đồ thị theo miền, đánh giá precision@3, dashboard tĩnh và bộ sinh PDF. Mọi tệp phụ đều được tạo lại từ notebook.",
        "Ba nguồn dữ liệu được mô tả bằng số dòng, số cột, khoảng thời gian, đơn vị và biến đích. Bảng được sinh trực tiếp từ DataFrame sau xử lý nên khớp với lần chạy hiện tại."], "data_profile.png"),
    ("4.2. KNOWLEDGE GRAPH – KNOWLEDGE BASE", [
        "Knowledge Base trong bài gồm hai lớp. Lớp văn bản giữ các đoạn mô tả dữ liệu, phương pháp và kết quả theo từng miền. Lớp đồ thị giữ thực thể cùng các quan hệ có hướng. Hai lớp dùng chung nhãn domain để ngăn truy xuất nhầm giữa tiểu đường, cổ phiếu và vàng.",
        f"Đồ thị tại lần chạy này có {kg.graph.number_of_nodes()} nút và {kg.graph.number_of_edges()} cạnh. Kích thước nhỏ giúp người đọc kiểm tra thủ công. Khi mở rộng, cùng schema có thể chuyển sang Neo4j hoặc một triple store mà không thay đổi ý nghĩa bộ ba.",
        "Knowledge Base không chỉ chứa kiến thức tĩnh. Các nút quan sát như giá FPT hoặc giá vàng gần nhất được tạo từ dữ liệu tải về. Cách làm này nối câu trả lời với đúng trạng thái thị trường tại thời điểm chạy."], "knowledge_graph.png"),
    ("4.2.1. BỘ BA, THỰC THỂ VÀ QUAN HỆ", [
        "Một bộ ba có dạng (s, p, o), trong đó s là thực thể nguồn, p là quan hệ có tên và o là thực thể hoặc giá trị đích. Ví dụ: (MA20, MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA, FPT.VN). Bộ ba này không khẳng định xu hướng sẽ tiếp diễn; nó chỉ diễn tả vai trò của chỉ báo.",
        "Thực thể được gắn type như Condition, Biomarker, Stock, TechnicalIndicator, Commodity hoặc MacroFactor. Thuộc tính domain tách ba không gian nghiệp vụ. Tên quan hệ dùng động từ viết hoa để người đọc có thể đọc thành câu.",
        "Việc chọn quan hệ là công đoạn có tính chủ quan. Báo cáo ưu tiên từ CÓ_THỂ_TÁC_ĐỘNG hoặc CÓ_QUAN_HỆ... thay vì GÂY_RA khi dữ liệu không đủ để chứng minh nhân quả."], None),
    ("4.2.2. KNOWLEDGE GRAPH CORE ENGINE", [
        "Lớp KnowledgeGraphCore bọc MultiDiGraph của NetworkX. add_entity quản lý nút và thuộc tính; add_relation thêm cạnh có hướng; neighbors trả về bằng chứng một bước; multi_hop tìm các đường đơn trong giới hạn số hop. Cấu trúc MultiDiGraph cho phép hai thực thể có nhiều loại quan hệ.",
        "Độ phức tạp của truy vấn lân cận tỷ lệ với bậc của nút. Truy vấn mọi đường đơn có thể tăng nhanh theo kích thước đồ thị, vì vậy max_hops phải được giới hạn. Với đồ thị sản xuất, cần chỉ mục, phân quyền và chiến lược truy vấn cụ thể thay vì duyệt toàn bộ.",
        "Toàn bộ cạnh được xuất thành knowledge_graph_triples.csv. Đây là điểm kiểm tra thuận tiện: mỗi dòng trong tệp phải đọc được như một phát biểu, không có nút mồ côi vô nghĩa và không chứa con số không rõ ngày."], "code_kg_core.png"),
    ("4.2.3. RDF VÀ LABELED PROPERTY GRAPH", [
        "RDF phù hợp khi cần URI toàn cục, liên kết dữ liệu và suy luận theo chuẩn W3C. Labeled Property Graph thuận tiện khi cạnh cần thuộc tính và truy vấn đường đi là trọng tâm. Bài chọn mô hình thứ hai vì mục tiêu là nguyên mẫu phân tích và trực quan hóa bằng Python.",
        "Nếu chuyển sang RDF, các entity_id cần ánh xạ thành IRI; relation thành predicate; giá trị số thành literal có datatype. Nếu chuyển sang Neo4j, type có thể thành node label, relation thành relationship type và các trường domain/source thành property.",
        "Không có mô hình nào tốt tuyệt đối. Tiêu chí chọn là khả năng tích hợp với hệ thống, yêu cầu suy luận, quy mô và kỹ năng vận hành của nhóm."], None),
    ("4.2.4. CHẤT LƯỢNG VÀ QUẢN TRỊ TRI THỨC", [
        "Một KG hữu ích cần hơn số lượng cạnh. Bốn kiểm tra tối thiểu gồm: tính duy nhất của định danh, tính hợp lệ của miền–phạm vi, nguồn gốc quan hệ và thời hạn của dữ liệu động. Các quan hệ y tế và tài chính phải có mức tin cậy hoặc câu chữ thể hiện độ bất định.",
        "Bài hiện lưu source ở lớp tài liệu và domain ở lớp đồ thị. Bản sản xuất nên bổ sung created_at, valid_from, valid_to, confidence và evidence_id cho từng cạnh. Khi đó, một quan hệ đã cũ có thể bị loại trước khi đưa vào prompt.",
        "Kiểm soát chất lượng cần vòng phản hồi của chuyên gia. Bác sĩ xác nhận ontology y tế; chuyên viên tài chính rà soát khái niệm và đơn vị; kỹ sư dữ liệu kiểm tra schema cùng pipeline cập nhật."], None),
    ("4.3. RAG — RETRIEVAL-AUGMENTED GENERATION", [
        "Pipeline RAG gồm ingestion, indexing, retrieval và generation. Văn bản được chia thành các đơn vị có nguồn; TF-IDF biến chúng thành vector; cosine similarity xếp hạng theo câu hỏi. Bước sinh ghép hai đoạn phù hợp nhất với các bộ ba liên quan.",
        "TF-IDF được chọn vì minh bạch, nhẹ và tái lập. Hạn chế là không hiểu đồng nghĩa sâu như embedding. Để giảm nhược điểm, câu hỏi được nhận diện miền bằng một từ điển alias và GraphRAG bổ sung các quan hệ từ đồ thị.",
        "Mỗi câu trả lời trả về cả documents, score, triples và domain. Cấu trúc này quan trọng hơn phần câu chữ: giao diện hoặc báo cáo có thể trình bày bằng chứng mà không phải phân tích ngược chuỗi văn bản."], "architecture.png"),
    ("4.3.1. NẠP, LẬP CHỈ MỤC VÀ TRUY XUẤT", [
        "Các tài liệu ngắn được xây từ mô tả bộ dữ liệu và kết quả thực nghiệm. Trong hệ thống lớn, ingestion cần đọc PDF, HTML hoặc cơ sở dữ liệu; chuẩn hóa Unicode; bỏ trùng; chia đoạn theo ngữ nghĩa; và gắn metadata trước khi embedding.",
        "TfidfVectorizer dùng unigram và bigram. Bigram giúp phân biệt các cụm như giá vàng, tiểu đường, lãi suất Fed. top_k=3 giữ ngữ cảnh ngắn và dễ kiểm tra. Precision@3 theo domain được dùng như phép đo đơn giản cho độ đúng của truy xuất.",
        "Điểm cosine chỉ phản ánh độ gần từ vựng, không phải độ đúng. Vì vậy báo cáo không diễn giải score như xác suất và không tự động loại bỏ nguồn chuyên môn chỉ vì score thấp."], "code_retriever.png"),
    ("4.3.2. TRUY XUẤT ĐỒ THỊ VÀ SUY LUẬN NHIỀU BƯỚC", [
        "graph_evidence nhận diện miền, chọn các nút cùng domain hoặc trùng từ khóa, rồi lấy quan hệ vào và ra. Phương án này đủ cho đồ thị nhỏ. Khi đồ thị lớn, cần entity linking, tìm kiếm theo embedding của nhãn và truy vấn Cypher có giới hạn.",
        "Suy luận nhiều bước được minh họa bởi chuỗi Rủi ro địa chính trị → Nhu cầu trú ẩn → Vàng quốc tế. Vector search có thể tìm một đoạn nói về địa chính trị nhưng không mặc nhiên biểu diễn hai quan hệ riêng. Đồ thị giúp giữ cấu trúc của lời giải thích.",
        "Đường đi chỉ là bằng chứng cấu trúc, không tự chứng minh nhân quả. Mọi kết luận vẫn phụ thuộc chất lượng cạnh và nguồn đã dùng để tạo cạnh."], "code_graphrag.png"),
    ("4.3.3. TỔNG HỢP CÂU TRẢ LỜI CÓ CĂN CỨ", [
        "Bộ sinh trong bài là deterministic synthesizer. Nó chọn hai tài liệu phù hợp, nối tối đa ba quan hệ đồ thị và thêm cảnh báo theo miền. Kết quả ít tự nhiên hơn LLM nhưng có lợi thế là chạy offline và không tự thêm sự kiện ngoài bằng chứng.",
        "Có thể thay bộ sinh bằng một LLM cục bộ hoặc dịch vụ API. Khi đó prompt phải tách rõ QUESTION, TEXT EVIDENCE, GRAPH EVIDENCE và yêu cầu từ chối nếu bằng chứng thiếu. Đầu ra nên ở dạng JSON để kiểm tra citation_id.",
        "Thiết kế production cần chống prompt injection từ tài liệu, lọc dữ liệu nhạy cảm, theo dõi phiên bản chỉ mục và ghi log các nguồn đã đưa cho mô hình."], None),
    ("4.3.4. ĐÁNH GIÁ RAG VÀ GRAPHRAG", [
        "Bộ kiểm thử gồm sáu câu hỏi, hai câu cho mỗi miền. vector_precision@3 đo tỷ lệ tài liệu thuộc đúng miền trong ba kết quả đầu. graph_evidence ghi nhận hệ thống có tìm được cạnh liên quan hay không. Điểm tổng hợp cộng một phần thưởng nhỏ khi cả hai kênh đều có bằng chứng.",
        f"Ở lần chạy này, precision@3 trung bình là {evaluation_df['vector_precision@3'].mean():.3f}; tỷ lệ câu hỏi có bằng chứng đồ thị là {evaluation_df['graph_evidence'].mean():.3f}; điểm bằng chứng lai là {evaluation_df['hybrid_evidence_score'].mean():.3f}.",
        "Đây là smoke test, chưa phải benchmark khoa học. Một đánh giá đầy đủ cần nhiều câu hỏi do người độc lập soạn, đáp án chuẩn, chấm faithfulness, answer relevance, citation correctness, độ trễ và chi phí."], "rag_benchmark.png"),
    ("4.4. KB VÀ RAG CHO BỆNH TIỂU ĐƯỜNG", [
        "Bộ dữ liệu diabetes của scikit-learn có 442 quan sát và 10 biến đầu vào đã chuẩn hóa. Biến đích là chỉ số định lượng mức tiến triển bệnh sau một năm. Vì các biến đã được biến đổi, kết quả không nên được đọc như ngưỡng lâm sàng trực tiếp.",
        "Random Forest được dùng để minh họa cách kết quả mô hình trở thành tài liệu và nút tri thức. Dữ liệu được chia 80/20 với random_state cố định. Mục tiêu ở đây là provenance của pipeline, không phải tối ưu leaderboard.",
        f"Tuning dùng GridSearchCV 5-fold trên tập train; tập test 20% được giữ riêng đến bước cuối. Tham số tốt nhất: {best_params}. CV MAE = {diabetes_metrics['CV_MAE']:.2f}; test MAE = {diabetes_metrics['MAE']:.2f}; test R² = {diabetes_metrics['R2']:.3f}."], "kg_diabetes.png"),
    ("4.4.1. ONTOLOGY VÀ XỬ LÝ DỮ LIỆU Y TẾ", [
        "Ontology tối giản gồm Condition, Biomarker, RiskFactor, Lifestyle, Monitoring và Professional. Các quan hệ như LIÊN_QUAN_ĐẾN hoặc HỖ_TRỢ_QUẢN_LÝ được chọn để tránh biến một tương quan thành chỉ định điều trị.",
        "Dữ liệu không có tên bệnh nhân hay định danh cá nhân. Trong tình huống thực, hồ sơ y tế cần mã hóa, phân quyền theo vai trò, nhật ký truy cập và quy trình chấp thuận. Không nên đưa thông tin nhận dạng vào dịch vụ embedding công cộng khi chưa có cơ sở pháp lý.",
        "Mô hình Random Forest được chọn sau GridSearchCV 5-fold. Việc tuning chỉ nhìn tập train; tập test giữ riêng để tránh đánh giá lạc quan do rò rỉ dữ liệu."], "code_diabetes_training.png"),
    ("4.4.2. THỰC NGHIỆM HỎI ĐÁP Y TẾ", [
        "Câu hỏi mẫu hỏi các yếu tố liên quan đến quản lý bệnh tiểu đường. Vector retriever tìm mô tả theo dõi đường huyết, HbA1c, huyết áp, cân nặng, ăn uống và vận động. Đồ thị bổ sung các cạnh nối bệnh với theo dõi và chuyên gia.",
        f"Câu trả lời sinh ra trong lần chạy: {demo_answers[0]['answer']}",
        "Biểu đồ đối chiếu giá trị thực - dự đoán và feature importance cho thấy mức độ phù hợp cũng như biến nào được mô hình sử dụng nhiều nhất."], "diabetes_model.png"),
    ("4.4.3. GIỚI HẠN VÀ HƯỚNG MỞ RỘNG Y TẾ", [
        "Bộ dữ liệu nhỏ, không đại diện cho dân số Việt Nam và không chứa bối cảnh điều trị. Chỉ số đích mang ý nghĩa nghiên cứu, khác với chẩn đoán đái tháo đường trong thực hành. Vì vậy không thể suy rộng độ chính xác mô hình thành hiệu quả lâm sàng.",
        "Hướng mở rộng là dùng dữ liệu đã khử định danh, xác định rõ biến kết cục, đánh giá theo nhóm tuổi/giới và mời chuyên gia lâm sàng kiểm tra ontology. Ngoài độ chính xác, cần đo calibration, fairness và khả năng giải thích.",
        "Với RAG y tế, mọi đoạn văn cần version và ngày hiệu lực. Hướng dẫn cũ phải được thu hồi khỏi chỉ mục thay vì chỉ thêm tài liệu mới."], None),
    ("4.5. KB VÀ RAG CHO CHỨNG KHOÁN", [
        f"Dữ liệu FPT.VN được lấy từ {market_sources['stock']}. Khoảng thời gian {stock_summary['start']}–{stock_summary['end']} có {stock_summary['observations']} quan sát hợp lệ. Giá đóng cửa gần nhất là {stock_summary['latest_close']:,.0f} {stock_summary['currency']}.",
        f"Lợi suất toàn kỳ là {stock_summary['period_return_pct']:.2f}%, biến động năm hóa {stock_summary['volatility_pct']:.2f}%, RSI14 {stock_summary['rsi14']:.1f}. Các số liệu phản ánh dữ liệu quá khứ và thay đổi khi chạy lại.",
        "Knowledge Graph gắn FPT.VN với giá gần nhất, MA20, MA60, RSI, thanh khoản, kết quả kinh doanh và tâm lý thị trường. Quan hệ được chọn như một khung giải thích, không phải mô hình dự báo giá."], "kg_stock.png"),
    ("4.5.1. TIỀN XỬ LÝ VÀ CHỈ BÁO FPT", [
        "Dữ liệu OHLCV được sắp xếp theo ngày; dòng thiếu Close bị loại. Return_1D là phần trăm thay đổi giá đóng cửa; MA20 và MA60 là trung bình trượt; volatility20 là độ lệch chuẩn 20 phiên quy đổi theo căn 252; RSI14 dùng mức tăng/giảm trung bình 14 phiên.",
        f"Tại điểm cuối, MA20 = {stock_summary['ma20']:,.0f}, MA60 = {stock_summary['ma60']:,.0f}. Việc so sánh giá với hai đường này chỉ mô tả trạng thái xu hướng, không đủ để xác nhận tín hiệu giao dịch.",
        "Khối lượng giao dịch từ nguồn miễn phí có thể bị điều chỉnh hoặc thiếu. Nếu dùng cho quyết định thực, cần đối chiếu dữ liệu chính thức từ sở giao dịch hoặc nhà cung cấp được cấp phép."], "fpt_market.png"),
    ("4.5.2. HỎI ĐÁP CHỨNG KHOÁN VỚI GRAPHRAG", [
        f"Câu trả lời mẫu: {demo_answers[1]['answer']}",
        "Ở đây, tài liệu mang số liệu còn đồ thị diễn giải vai trò của chỉ báo. Nếu chỉ dùng vector search, hệ thống có thể trả đúng MA20 nhưng bỏ mất việc MA20 là chỉ báo ngắn hạn của FPT.VN. Nếu chỉ dùng đồ thị, hệ thống lại thiếu giá trị số mới nhất.",
        "Cách ghép hai kênh làm rõ ranh giới giữa fact động và knowledge tĩnh. Đây cũng là lý do report ghi thời gian chạy và nguồn của chuỗi dữ liệu."], "code_market_data.png"),
    ("4.5.3. RỦI RO KHI DIỄN GIẢI KẾT QUẢ", [
        "Survivorship bias, corporate actions, thanh khoản và cách điều chỉnh giá có thể làm sai phân tích. Yahoo Finance thuận tiện cho bài tập nhưng không phải nguồn duy nhất để kiểm chứng giao dịch tại Việt Nam.",
        "Không có backtest chiến lược trong phạm vi bài. Việc nhìn thấy MA20 vượt MA60 sau khi sự kiện đã xảy ra không chứng minh lợi nhuận tương lai. Một nghiên cứu nghiêm túc cần quy tắc vào/ra, phí, trượt giá, tập kiểm thử ngoài mẫu và kiểm soát data leakage.",
        "Câu trả lời của hệ thống luôn kết thúc bằng cảnh báo không phải khuyến nghị mua bán."], None),
    ("4.6. KB VÀ RAG CHO GIÁ VÀNG", [
        f"Dữ liệu hợp đồng vàng GC=F được lấy từ {market_sources['gold']}. Khoảng {gold_summary['start']}–{gold_summary['end']} có {gold_summary['observations']} quan sát; giá gần nhất {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz.",
        f"Lợi suất toàn kỳ {gold_summary['period_return_pct']:.2f}%, biến động năm hóa {gold_summary['volatility_pct']:.2f}%, RSI14 {gold_summary['rsi14']:.1f}. GC=F là tham chiếu quốc tế, không đồng nhất với giá vàng miếng SJC trong nước.",
        "Đồ thị mô tả lãi suất Fed, USD, lạm phát, địa chính trị và nhu cầu trú ẩn. Các cạnh dùng từ thường gặp hoặc có thể vì độ mạnh, thậm chí chiều quan hệ có thể thay đổi giữa các chế độ thị trường."], "kg_gold.png"),
    ("4.6.1. CHỈ BÁO VÀ KNOWLEDGE GRAPH VÀNG", [
        f"MA20 hiện là {gold_summary['ma20']:,.2f}, MA60 là {gold_summary['ma60']:,.2f}. Hai đường được tính trực tiếp từ Close. Không có nội suy cho ngày nghỉ; chuỗi giữ nguyên lịch phiên của nguồn.",
        "Chuỗi Rủi ro địa chính trị → Nhu cầu trú ẩn → Vàng quốc tế là ví dụ multi-hop. Chuỗi Vàng quốc tế → Giá vàng SJC chỉ là quan hệ tham chiếu; để giải thích chênh lệch trong nước cần thêm tỷ giá, thuế, cung cầu và cơ chế quản lý.",
        "Một bản mở rộng có thể nhập FRED cho lãi suất thực và DXY, sau đó gắn mỗi quan sát với timestamp để thực hiện truy vấn theo giai đoạn."], "gold_market.png"),
    ("4.6.2. HỎI ĐÁP VĨ MÔ VỚI GRAPHRAG", [
        f"Câu trả lời mẫu: {demo_answers[2]['answer']}",
        "Truy xuất văn bản cung cấp cảnh báo rằng quan hệ thay đổi theo giai đoạn. Đồ thị cung cấp các cạnh nghịch thường gặp và đường đi qua nhu cầu trú ẩn. Hai phần cùng xuất hiện giúp câu trả lời bớt tuyệt đối hóa.",
        "Nếu triển khai với LLM, prompt cần cấm mô hình tự chèn mục tiêu giá và xác suất tăng giảm khi tài liệu không có số liệu hỗ trợ."], None),
    ("4.6.3. GIỚI HẠN DỮ LIỆU VÀNG", [
        "Hợp đồng tương lai có ngày đáo hạn và cơ chế nối chuỗi; dữ liệu GC=F do nhà cung cấp tổng hợp. Giá spot XAU/USD, futures và SJC là ba sản phẩm khác nhau. Không nên trộn đơn vị USD/oz với VND/lượng nếu chưa quy đổi và ghi rõ phí/chênh lệch.",
        "Bài chưa mô hình hóa tỷ giá USD/VND, lãi suất thực hay vị thế quỹ ETF. Vì vậy đồ thị vĩ mô mang tính khung tri thức, chưa phải mô hình định lượng nhân quả.",
        "Hướng mở rộng là kết hợp dữ liệu có timestamp, kiểm tra lag giữa biến vĩ mô và giá vàng, rồi lưu kết quả kiểm định cùng confidence trên cạnh."], None),
    ("4.7. DEPLOYMENT", [
        "Notebook sinh dashboard tĩnh trong vercel_app. index.html nhúng trực tiếp các hình dưới dạng base64, do đó toàn bộ giao diện chỉ cần một tệp HTML cùng vercel.json. Vercel có thể phục vụ nội dung qua CDN mà không cần chạy Python trên máy chủ.",
        "Luồng triển khai gồm: Run All notebook → kiểm tra PDF và dashboard → chạy npx vercel --prod trong vercel_app → nhận URL. Nếu đặt VERCEL_TOKEN, ô cuối có thể chạy lệnh này tự động. Token không được ghi vào notebook hoặc PDF.",
        f"Dashboard hiển thị số nút/cạnh, chỉ số mô hình, giá gần nhất, ba câu hỏi mẫu, đồ thị và benchmark. Mã nguồn tại <link href='{GITHUB_URL}' color='#2563eb'>{GITHUB_URL}</link>; bản web tại <link href='{DEPLOY_URL}' color='#2563eb'>{DEPLOY_URL}</link>."], "code_deployment.png"),
    ("4.7.1. KIẾN TRÚC TRIỂN KHAI VÀ TÁI LẬP", [
        "Bản triển khai tách compute và presentation. Compute chạy trong notebook; presentation là static site. Cách tách này phù hợp Vercel và giảm rủi ro cài thư viện nặng trên serverless function. Nhược điểm là muốn cập nhật dữ liệu phải chạy lại notebook và deploy lại.",
        "Để chuyển thành ứng dụng động, có thể dùng FastAPI cho endpoint /ask, Neo4j cho đồ thị, vector database cho tài liệu và một scheduler để cập nhật giá. Khi đó cần cache, rate limit, monitoring và secret management.",
        "Tính tái lập được bảo đảm bởi seed cố định, tệp CSV đã xử lý, metadata nguồn, chỉ số tại thời điểm chạy và việc PDF được tạo sau cùng từ cùng các biến trong bộ nhớ."], None),
    ("4.7.2. BẢO MẬT, QUYỀN RIÊNG TƯ VÀ VẬN HÀNH", [
        "Static dashboard không lưu thông tin người dùng. Nếu thêm chức năng hỏi đáp, đầu vào phải được lọc, log phải tránh dữ liệu nhạy cảm và câu trả lời cần giới hạn theo vai trò. Đối với y tế, cần quy trình xử lý dữ liệu cá nhân và phê duyệt phù hợp.",
        "Nguồn tài liệu ngoài có thể chứa prompt injection. Pipeline production nên xem nội dung truy xuất là dữ liệu không tin cậy, không cho phép nó thay đổi system instruction hoặc yêu cầu tiết lộ secret.",
        "Các chỉ số vận hành nên gồm độ trễ truy xuất, tỷ lệ không có bằng chứng, citation coverage, lỗi tải dữ liệu và độ mới của chỉ mục. Cần rollback được phiên bản KG và tài liệu."], None),
    ("4.7.3. KIỂM THỬ TRIỂN KHAI", [
        "Ba lớp kiểm thử được đề xuất: unit test cho chỉ báo và bộ ba; integration test cho toàn pipeline; smoke test mở index.html và PDF. Notebook hiện thực smoke test bằng cách xác nhận tệp tồn tại, số trang PDF và kích thước đầu ra sau khi xây dựng.",
        "Trước khi nộp, người thực hiện cần mở PDF để kiểm tra font tiếng Việt, hình không vỡ và bảng không tràn. Với web, kiểm tra màn hình điện thoại, liên kết và cảnh báo y tế/tài chính.",
        "Deployment được coi là hoàn thành khi URL production truy cập được, không chứa token và số liệu khớp bản PDF cùng lần chạy."], None),
    ("ĐÁNH GIÁ TỔNG HỢP", [
        "Kết quả cho thấy một notebook có thể bao quát tải dữ liệu, tiền xử lý, KG, RAG, đánh giá, báo cáo và web. GraphRAG lai cung cấp thêm loại bằng chứng mà Vector RAG không có: quan hệ có tên và đường đi giữa các thực thể.",
        "Đổi lại, KG yêu cầu thiết kế ontology, kiểm soát quan hệ và cập nhật theo thời gian. Chi phí này chỉ đáng bỏ ra khi bài toán cần giải thích, liên kết đa bước hoặc quản trị tri thức dài hạn.",
        "Bài làm đạt mục tiêu nguyên mẫu tái lập. Hạn chế chính là corpus nhỏ, đánh giá ít câu hỏi, generator dạng mẫu và chưa có chuyên gia miền thẩm định. Những giới hạn này được ghi rõ thay vì suy rộng kết quả."], "rag_benchmark.png"),
    ("KẾT LUẬN", [
        "Knowledge Graph và RAG bổ trợ nhau theo cách thực dụng: văn bản đem lại ngữ cảnh linh hoạt, đồ thị đem lại cấu trúc và provenance. Trên ba miền thử nghiệm, cùng một pipeline vẫn giữ được nguồn, thời điểm và cảnh báo riêng.",
        "Điều quan trọng nhất rút ra là không nên coi GraphRAG như một nhãn công nghệ thay thế cho thiết kế dữ liệu. Chất lượng câu trả lời bắt đầu từ việc định nghĩa thực thể, quan hệ, nguồn và phạm vi kết luận.",
        "Notebook kèm theo là sản phẩm có thể chạy lại. PDF và dashboard chỉ là hai cách trình bày cùng một trạng thái kết quả, nhờ đó giảm sai lệch giữa code và báo cáo."], None),
]

for title, paragraphs, figure in pages:
    story.append(P(title, "VNChapter" if title.count(".") <= 1 else "VNHeading"))
    for paragraph in paragraphs:
        story.append(P(paragraph))
    if figure:
        img_path = FIG_DIR / figure
        if img_path.exists():
            img = Image(str(img_path), width=15.5*cm, height=8.2*cm)
            img._restrictSize(15.5*cm, 11.0*cm)
            story += [Spacer(1, .15*cm), img, P(f"Hình: {title}", "VNCaption")]
    story.append(PageBreak())

# Bảng kết quả chi tiết
story += [P("PHỤ LỤC A — KẾT QUẢ TRUY XUẤT", "VNChapter")]
eval_data = [["Câu hỏi", "Miền", "P@3", "Graph", "Hybrid"]] + [
    [P(r["query"], "VNSmall"), r["expected_domain"], f"{r['vector_precision@3']:.2f}", str(r["graph_evidence"]), f"{r['hybrid_evidence_score']:.2f}"]
    for _, r in evaluation_df.iterrows()
]
story += [report_table(eval_data, [8.4*cm, 2.0*cm, 1.6*cm, 1.6*cm, 1.8*cm], font_size=7.5),
          Spacer(1, .5*cm), P("Bảng này là phép kiểm tra hồi quy cho pipeline. Nó không thay thế đánh giá câu trả lời bởi chuyên gia."), PageBreak()]

story += [P("PHỤ LỤC B — MẪU BỘ BA TRI THỨC", "VNChapter")]
triple_data = [["Subject", "Predicate", "Object"]] + [
    [P(str(r.Subject), "VNSmall"), P(str(r.Predicate), "VNSmall"), P(str(r.Object), "VNSmall")]
    for r in triples_df.head(20).itertuples()
]
story += [report_table(triple_data, [4.9*cm, 5.2*cm, 5.4*cm], font_size=7.2), PageBreak()]

references = [
    "[1] W3C, RDF 1.1 Concepts and Abstract Syntax, W3C Recommendation, 2014.",
    "[2] T. Berners-Lee, J. Hendler, O. Lassila, The Semantic Web, Scientific American, 2001.",
    "[3] K. Guu et al., REALM: Retrieval-Augmented Language Model Pre-Training, ICML, 2020.",
    "[4] P. Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks, NeurIPS, 2020.",
    "[5] D. Edge et al., From Local to Global: A Graph RAG Approach to Query-Focused Summarization, 2024.",
    "[6] scikit-learn developers, Diabetes dataset and TfidfVectorizer documentation.",
    "[7] NetworkX developers, MultiDiGraph documentation.",
    "[8] Yahoo Finance chart endpoint, mã FPT.VN và GC=F; truy cập tại thời điểm chạy notebook.",
    "[9] World Health Organization, Diabetes fact sheet; dùng cho bối cảnh quản lý bệnh, không dùng để chẩn đoán.",
    "[10] Vercel Documentation, Static Deployments and CLI.",
    f"[11] Mã nguồn bài làm: <link href='{GITHUB_URL}' color='#2563eb'>{GITHUB_URL}</link>.",
    f"[12] Web triển khai: <link href='{DEPLOY_URL}' color='#2563eb'>{DEPLOY_URL}</link>.",
]
story += [P("TÀI LIỆU THAM KHẢO", "VNChapter")]
story += [P(ref) for ref in references]
story += [Spacer(1, .5*cm), P(f"Báo cáo được sinh tự động từ notebook lúc {RUN_AT:%d/%m/%Y %H:%M:%S %Z}. Dữ liệu nguồn: FPT — {market_sources['stock']}; vàng — {market_sources['gold']}.", "VNSmall")]

doc = ReportDoc(str(REPORT_PATH))
doc.build(story)
print("Đã tạo báo cáo:", REPORT_PATH)

Đã tạo báo cáo: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/Tieuluan02_06_Nhom2_Giangkt.pdf


In [10]:
# 6. KIỂM TRA ĐẦU RA VÀ TRIỂN KHAI TÙY CHỌN
assert REPORT_PATH.exists() and REPORT_PATH.stat().st_size > 100_000
assert (WEB_DIR / "index.html").exists()
assert (DATA_DIR / "knowledge_graph_triples.csv").exists()

try:
    from pypdf import PdfReader
except Exception:
    bundled_site = Path("/home/jellalaz/.cache/codex-runtimes/codex-primary-runtime/dependencies/python")
    if bundled_site.exists() and str(bundled_site) not in sys.path:
        sys.path.append(str(bundled_site))
    from pypdf import PdfReader
page_count = len(PdfReader(str(REPORT_PATH)).pages)
print(f"PDF hợp lệ: {page_count} trang, {REPORT_PATH.stat().st_size/1024/1024:.2f} MB")
print("Web package:", WEB_DIR)

# Để deploy thực sự: đặt biến môi trường VERCEL_TOKEN và chuyển cờ thành True.
# Token tuyệt đối không được ghi trực tiếp vào notebook.
DEPLOY_TO_VERCEL = False
if DEPLOY_TO_VERCEL:
    token = os.getenv("VERCEL_TOKEN")
    if not token:
        raise RuntimeError("Thiếu biến môi trường VERCEL_TOKEN")
    subprocess.run(
        ["npx", "vercel", "--prod", "--yes", "--token", token],
        cwd=WEB_DIR, check=True
    )
else:
    print("Chưa deploy ra Internet. Chạy `cd vercel_app && npx vercel --prod` khi đã đăng nhập Vercel.")

PDF hợp lệ: 38 trang, 1.39 MB
Web package: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/vercel_app
Chưa deploy ra Internet. Chạy `cd vercel_app && npx vercel --prod` khi đã đăng nhập Vercel.
